In [2]:
import json
import matplotlib.pyplot as plt
import ipywidgets as widgets

# ---------- settings ----------
years_to_use = [2020, 2021, 2022, 2023, 2024, 2025, 2026]
england_population = 57000000   # approximate, check the figure for England (ONS)

months = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
month_names = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
               "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

files = {
    "positivity": {
        "COVID-19": "covid_positivity.json",
        "Influenza": "influenza_positivity.json",
        "RSV": "RSV_positivity.json",
    },
    "admissions": {
        "COVID-19": "covid_admissions.json",
        "Influenza": "influenza_admissions.json",
        "RSV": "RSV_cases.json",
    },
}

legend_labels = {
    "positivity": {
        "COVID-19": "COVID-19",
        "Influenza": "Influenza",
        "RSV": "RSV",
    },
    "admissions": {
        "COVID-19": "COVID-19 (converted from daily counts)",
        "Influenza": "Influenza",
        "RSV": "RSV",
    },
}

colours = {"COVID-19": "tab:pink", "Influenza": "tab:blue", "RSV": "tab:purple"}

y_labels = {
    "positivity": "Average PCR positivity (%)",
    "admissions": "Average weekly hospital admission rate per 100,000",
}
titles = {
    "positivity": "Average Monthly PCR positivity (2020 - 2026)",
    "admissions": "Average Monthly Hospital Admissions (2020 - 2026)",
}

# ---------- work out the averages ----------
def month_averages(filename):
    with open(filename) as f:
        records = json.load(f)

    totals = {}
    counts = {}
    for month in months:
        totals[month] = 0
        counts[month] = 0

    for record in records:
        if record["year"] in years_to_use:
            month = record["month"]
            totals[month] = totals[month] + record["metric_value"]
            counts[month] = counts[month] + 1

    averages = []
    for month in months:
        if counts[month] > 0:
            averages.append(totals[month] / counts[month])
        else:
            averages.append(0)
    return averages

data = {}
for metric in files:
    data[metric] = {}
    for disease in files[metric]:
        data[metric][disease] = month_averages(files[metric][disease])

# convert COVID-19 daily counts into a weekly rate per 100,000
# (keep this in the same cell as the loop above, or it converts twice)
for i in range(len(months)):
    daily = data["admissions"]["COVID-19"][i]
    data["admissions"]["COVID-19"][i] = daily * 7 / england_population * 100000

# ---------- draw the chart ----------
def draw_chart(metric):
    diseases = list(files[metric])
    width = 0.8 / len(diseases)

    plt.figure(figsize=(12, 5))
    n = 0
    for disease in diseases:
        positions = []
        for i in range(len(months)):
            positions.append(i + n * width)
        plt.bar(positions, data[metric][disease], width=width,
                label=legend_labels[metric][disease], color=colours[disease])
        n = n + 1

    centre = []
    for i in range(len(months)):
        centre.append(i + width * (len(diseases) - 1) / 2)
    plt.xticks(centre, month_names)

    plt.xlabel("Month")
    plt.ylabel(y_labels[metric])
    plt.title(titles[metric])
    plt.ylim(bottom=0)
    plt.grid(axis="y", alpha=0.3)
    plt.legend()
    plt.show()

# ---------- the toggle ----------
toggle = widgets.ToggleButtons(
    options=[("Positive tests", "positivity"), ("Hospital admissions", "admissions")],
    description="Chart:",
)

widgets.interact(draw_chart, metric=toggle)

interactive(children=(ToggleButtons(description='Chart:', options=(('Positive tests', 'positivity'), ('Hospita…

<function __main__.draw_chart(metric)>

Bar charts show: 
- average hospital admissions for each month from 2020 - 2026, for COVID-19, Influenza and RSV.
- average PCR positive tests for each month from 2020 - 2026, for COVID-19, Influenza and RSV.
Trend shows all are more prevalent in winter but COVID-19 follows trend less strictly.